In [1]:
# os gives us functions to work with files and folders (check paths, change directory, create folders)
import os

# %pwd is a Jupyter "magic" command: prints the Present Working Directory.
# A notebook starts in the folder where it lives, so this prints ".../ML-flow/research"
%pwd

'/Users/strava/Developer/personal/ml-project/ML-flow/research'

In [2]:
# Move one level up, from "research/" to the project root "ML-flow/".
# Why? Paths like "config/config.yaml" and "artifacts/..." are written relative to the
# project root, so the notebook must run from there or it won't find those files.
os.chdir("../")

# Confirm the move: should now print ".../ML-flow"
%pwd

'/Users/strava/Developer/personal/ml-project/ML-flow'

In [3]:
# dataclass auto-generates __init__, __repr__ etc. for classes that only hold data
from dataclasses import dataclass
# Path is a cleaner, cross-platform way to represent file/folder paths than plain strings
from pathlib import Path

# frozen=True makes the object read-only after creation, so settings can't be
# changed by accident somewhere else in the code.
@dataclass(frozen=True)
class DataIngestionConfig:
    """Holds all the settings needed by the data ingestion step (values come from config.yaml)."""
    root_dir: Path           # main folder for ingestion output, e.g. artifacts/data_ingestion
    source_URL: str          # web address the dataset zip is downloaded from
    local_data_file: Path    # where the downloaded zip is saved, e.g. artifacts/data_ingestion/data.zip
    unzip_dir: Path          # folder the zip contents are extracted into

In [4]:
# Import everything from the constants module: CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
# (these are the locations of config.yaml, params.yaml and schema.yaml)
from wineQualityML.constants import *

# Our own helper functions:
#   read_yaml          -> reads a .yaml file and returns its content as an easy-to-access object
#   create_directories -> creates the given folders if they don't already exist
from wineQualityML.utils.common import read_yaml, create_directories

In [5]:
class ConfigurationManager:
    """Reads the YAML files and hands out ready-to-use config objects for each pipeline step."""

    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,   # default: config/config.yaml
        params_filepath = PARAMS_FILE_PATH,   # default: params.yaml (model hyperparameters)
        schema_filepath = SCHEMA_FILE_PATH):  # default: schema.yaml (expected data columns/types)

        # Load each YAML file into memory
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        # Make sure the top-level "artifacts" folder exists (all pipeline outputs go inside it)
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        # Pick out just the "data_ingestion" section of config.yaml
        config = self.config.data_ingestion

        # Create artifacts/data_ingestion so there is a place to save the download
        create_directories([config.root_dir])

        # Copy the YAML values into a DataIngestionConfig object
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

        return data_ingestion_config

In [6]:
import os
# urllib.request lets Python download files from a web URL
import urllib.request as request
# zipfile lets Python open and extract .zip archives
import zipfile
# Our project logger: writes messages to the console and the log file instead of using print()
from wineQualityML import logger
# Helper that returns a human-readable file size (e.g. "~ 90 KB")
from wineQualityML.utils.common import get_size

In [7]:
class DataIngestion:
    """Does the actual work: downloads the dataset zip and extracts it."""

    def __init__(self, config: DataIngestionConfig):
        # Keep the settings so the methods below can use them
        self.config = config

    def download_file(self):
        # Only download if the zip isn't already on disk (avoids re-downloading every run)
        if not os.path.exists(self.config.local_data_file):
            # urlretrieve downloads from the URL and saves straight to the given file path.
            # It returns the saved filename and the HTTP response headers.
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            # Log what was saved plus the server's response info (size, type, date...)
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            # File is already there: just log its size and skip the download
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")

    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        # Create the destination folder if needed; exist_ok=True means no error if it exists
        os.makedirs(unzip_path, exist_ok=True)
        # Open the zip in read mode ('r'); the with-block closes it automatically afterwards
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            # Unpack every file in the archive into the destination folder
            zip_ref.extractall(unzip_path)

In [9]:
# Run the whole ingestion step, this is the data ingestion pipeline
try:
    config = ConfigurationManager()                                    # 1. read the YAML files
    data_ingestion_config = config.get_data_ingestion_config()         # 2. build the ingestion settings
    data_ingestion = DataIngestion(config=data_ingestion_config)       # 3. create the worker object
    data_ingestion.download_file()                                     # 4. download the zip (if not already there)
    data_ingestion.extract_zip_file()                                  # 5. unzip it into artifacts/data_ingestion
except Exception as e:
    # Re-raise so the real error and traceback show up in the notebook
    raise e

[2026-10-04 11:44:53,495: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-10-04 11:44:53,497: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-04 11:44:53,497: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-10-04 11:44:53,498: INFO: common: created directory at: artifacts]
[2026-10-04 11:44:53,499: INFO: common: created directory at: artifacts/data_ingestion]
[2026-10-04 11:44:54,374: INFO: 3897176636: artifacts/data_ingestion/data.zip download! with following info: 
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: EDB6:0BB9:65870:D03E9:6AC1EEDD
x-github-edge-region: southeastasia
Ac